# Astrology Notebook (thin UI)

この Notebook は **GitHub 上の `astrology.py` を正本**として読み込み、実行する UI です。  
ロジック本体は `astrology.py` 側に集約し、Notebook は入力設定・実行・表示・ダウンロードに専念します。


> Synastry文は `astrology.py` 側の専用文章エンジン（6セクション構成）をそのまま利用します。Notebook単体に旧テンプレートを持たないため、表示品質は常に `.py` と同期されます。


In [ ]:
# Cell 1: 初期セットアップ（Colab想定）
import os
from pathlib import Path

REPO_URL = "https://github.com/flareray0/astrology.git"
REPO_DIR = Path("/content/astrology")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("[INFO] repo exists -> git pull")
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
!pip install -q -r requirements.txt

os.environ.setdefault("RESULT_OUTPUT_DIR", str(REPO_DIR / "data" / "results"))
print("RESULT_OUTPUT_DIR=", os.environ["RESULT_OUTPUT_DIR"])


In [ ]:
# Cell 2: astrology.py の読み込み（再読み込みしやすく）
import importlib
import astrology

astrology = importlib.reload(astrology)
print("[OK] astrology module loaded:", astrology.__file__)

resolved_ephe = astrology.configure_ephemeris()
print("[EPHE] configured:", resolved_ephe)
astrology.print_ephemeris_status()


In [ ]:
# Cell 3: 設定ブロック（ここを編集）
chart_mode = "natal"  # natal / progressed / transit / triple / synastry
PERSON_NAME = "あなた"
PERSON2_NAME = "相手"

# natal
NATAL_DATE = (1984, 11, 15)
NATAL_TIME = "11:27"
NATAL_TZ = 9
NATAL_LAT = 37.38
NATAL_LON = 140.18

# person2 (synastry)
PERSON2_DATE = (1967, 5, 13)
PERSON2_TIME = "00:00"
PERSON2_TZ = 9
PERSON2_LAT = 35.68
PERSON2_LON = 139.65

# progressed
PROGRESS_DATE = (1984, 12, 26)
PROGRESS_TIME = "00:00"
PROGRESS_TZ = 9
PROGRESS_LAT = 37.38
PROGRESS_LON = 140.18

# transit
TRANSIT_DATE = (2026, 2, 12)
TRANSIT_TIME = "00:00"
TRANSIT_TZ = 9
TRANSIT_LAT = 37.38
TRANSIT_LON = 140.18

include_asteroids = True
include_minor_aspects = True
include_composite_aspects = True
hsys = "K"


In [ ]:
# Cell 4: 実行セル（chart_mode 全対応）
if chart_mode not in astrology.CHART_TYPE_LABELS:
    raise ValueError(f"unsupported chart_mode: {chart_mode}")

natal = astrology.build_chart_from_input(NATAL_DATE, NATAL_TIME, NATAL_TZ, NATAL_LAT, NATAL_LON, hsys=hsys, include_asteroids=include_asteroids)
progressed = astrology.build_chart_from_input(PROGRESS_DATE, PROGRESS_TIME, PROGRESS_TZ, PROGRESS_LAT, PROGRESS_LON, hsys=hsys, include_asteroids=include_asteroids)
transit = astrology.build_chart_from_input(TRANSIT_DATE, TRANSIT_TIME, TRANSIT_TZ, TRANSIT_LAT, TRANSIT_LON, hsys=hsys, include_asteroids=include_asteroids)
person2 = astrology.build_chart_from_input(PERSON2_DATE, PERSON2_TIME, PERSON2_TZ, PERSON2_LAT, PERSON2_LON, hsys=hsys, include_asteroids=include_asteroids)

result = astrology.run_report_by_mode(
    chart_mode,
    natal=natal,
    progressed=progressed,
    transit=transit,
    person2=person2,
    person_name=PERSON_NAME,
    person2_name=PERSON2_NAME,
    include_minor_aspects=include_minor_aspects,
    include_composite_aspects=include_composite_aspects,
)

print(result["interpretation"])
print("[SAVE]", result["result_path"], result["interpretation_path"])


In [ ]:
# Cell 5: ダウンロード（Colab）
from google.colab import files

files.download(result["result_path"])
files.download(result["interpretation_path"])
